<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **2. Cellular Networking**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 2.1 Introduction

A cellular network is a wide-area wireless system that divides geography into many small coverage regions (cells) served by base stations. Each cell reuses a finite pool of radio spectrum to connect many user devices at once, handing users over from one cell to the next as they move. This architecture allows massive spatial reuse of spectrum and continuous mobility while keeping radio links short enough to be efficient. Modern networks layer multiple frequency bands, technologies, and antenna configurations (from sub-6 GHz macro cells to mmWave small cells) to balance reach and capacity.

Planning is necessary because radio resources are scarce, propagation is variable, and demand is uneven. Spectrum must be allocated across cells to limit interference while maximizing reuse; transmit powers, antenna patterns, and site placements must be tuned so coverage is reliable at the edges without wasting energy or spilling interference onto neighbors. Capacity must match the expected load, which depends on user density, traffic mix, and peak events that can exceed everyday conditions by an order of magnitude. Physical realities - terrain, buildings, and clutter - distort signals, while regulatory constraints, backhaul, and cost shape what can be built. Good planning turns these constraints into a coherent design: cell sizes that meet both coverage and throughput targets, frequencies assigned to minimize contention, and a layout that can scale as usage grows.

This notebook walks through a coverage-and-capacity–driven approach to cellular network planning. We will use simple models you can tune with a few key inputs and visualize on a hexagonal grid over a city-sized area.


### 2.2 Theory

#### 2.2.1 Determining cell size based on coverage
The planning of a cellular network involves balancing coverage (based on power levels) and capacity (based on the number of clients and data requirements). Coverage is primarily determined by the transmit power $P_{\text{TX}}$, the receiver sensitivity $P_{\text{RX, min}}$, and the path loss $L_{\text{path}}$.

To determine the maximum size of a cell, the so-called link budget is needed. A link budget is a one-line accounting of all gains and losses from transmitter to receiver: power, antenna gains, path loss, and margins. It is used to predict the received signal level and reliability. The link budget equation relates the transmit power to the minimum received power:
$$
P_{\text{RX}} = P_{\text{TX}} + G_{\text{TX}} + G_{\text{RX}} - L_{\text{path}}
$$
where $P_{\text{RX}}$ is the received power at the cell edge (dBm) and $P_{\text{TX}}$ is the transmit power of the base station (dBm). $G_{\text{TX}}$ and $G_{\text{RX}}$ are the antenna gains of the transmitter and receiver (dBi), and $L_{\text{path}}$ determines the path loss (dB), which depends on distance, frequency, and environmental factors. The free-space path loss model below is an example for predicting how a radio signal weakens over a clear line-of-sight path due only to geometric spreading:
$$
L_{\text{path}} = 20\log_{10}(d) + 20\log_{10}(f) + 20\log_{10}\left(\frac{4\pi}{c}\right)
$$
where the parameter $d$ is the distance between transmitter and receiver in meters, $f$ determines the carrier frequency in Hertz, and $c$ is the speed of light in $\text{m/s}$. Rearranging to solve for $d_{\text{max}}$ (maximum cell radius):
$$
d_{\text{max}} = 10^{\frac{P_{\text{TX}} + G_{\text{TX}} + G_{\text{RX}} - P_{\text{RX, min}} - 20\log_{10}(f) - 20\log_{10}(4\pi/c)}{20}}
$$


The effect of the frequency on the cell size follows the same physical principles in indoor and outdoor environments. At low frequencies (below 1 GHz), signals propagate further outdoors and penetrate walls and obstacles better indoors, which makes them ideal for providing wide coverage in rural outdoor areas and reliable indoor coverage inside buildings with fewer base stations.

At mid frequencies (1-3 GHz), there is a balance: outdoors, the coverage range is smaller than at low bands but still sufficient for urban and suburban areas, while indoors the signals can usually penetrate walls and windows but with some loss, meaning extra indoor cells or repeaters may still be needed in dense buildings.

At high frequencies (3–6 GHz and especially millimeter-wave above 24 GHz), outdoor coverage becomes limited to short distances with a higher number of small cells required, and signals are easily blocked by obstacles like buildings, trees, and even rain. Indoors, high frequencies suffer significant attenuation through walls, glass, and furniture, so coverage is usually limited to the same room or open space unless dedicated indoor small cells or distributed antenna systems are installed.

In outdoor deployments, low frequency bands are preferred for wide-area coverage, but high frequency bands are still helpful to achieve high network capacity in dense zones. In indoor deployments, low frequency bands can penetrate from outdoor macro cells into buildings and are therefore preferred by operators, but high frequency bands generally provide the necessary network capacity. They require, however, dedicated indoor infrastructure to work effectively (for example, one access point per room).

#### 2.2.2 Adjusting the cell size based on capacity
Once coverage is determined, capacity considerations can shrink the cell size to accommodate more users or higher data rates. The capacity $C$ of a single cell is given by:
$$
C = N \cdot \log_2(1 + \text{SINR})
$$
where $N$ is the number of resource blocks or channels available, and $\text{SINR}$ is the Signal-to-Interference-and-Noise Ratio, defined as:
  $$
  \text{SINR} = \frac{P_{\text{signal}}}{P_{\text{interference}} + P_{\text{noise}}}
  $$

In this definition, $P_{\text{noise}} = k T B$ with $k$ as Boltzmann constant, $T$ defining the temperature in Kelvin, and $B$ defining the system bandwidth in Hertz. The total demand per cell $D_{\text{cell}}$ is:
$$
D_{\text{cell}} = U \cdot R_{\text{user}}
$$
where $U$ is the number of users in the cell, and $R_{\text{user}}$ is the average data rate required per user. To ensure sufficient capacity:
$$
D_{\text{cell}} \leq C
$$

If $D_{\text{cell}} > C$, the cell size needs to be reduced. The area of the cell is proportional to $d_{\text{max}}^2$, so reducing the cell size proportionally increases the capacity, as there would then be more cells in the same area.


#### 2.2.3 Combining coverage and capacity
The relationship between coverage and capacity is a trade-off. When the network is coverage-limited, there are large cells with fewer users, and $d_{\text{max}}$ is determined by $P_{\text{TX}}$ and $P_{\text{RX, min}}$. In capacity-limited, high user-density regions there are smaller cells, and $d_{\text{max}}$ is determined by the required capacity per cell.

  $$
  d_{\text{coverage}} = f(P_{\text{TX}}, P_{\text{RX, min}}, L_{\text{path}})
  $$
 
  $$
  d_{\text{capacity}} \propto \sqrt{\frac{C}{D_{\text{cell}}}}
  $$

By combining these equations, we can model the cell radius $d_{\text{cell}}$ dynamically as:
$$
d_{\text{cell}} = \min(d_{\text{coverage}}, d_{\text{capacity}})
$$



### 2.3 Cellular network planning for the city of Zurich

The following model visualizes the relationship between coverage and capacity with hexagonal cells. The grid visualizes cells with capacity-limited and with coverage-limited radius. The color-code helps to distinguish coverage-limited vs. capacity-limited regions based on the equations provided for $d_{\text{coverage}}$ and $d_{\text{capacity}}$, see above. We will later modify certain parameters and analyze their implications for the network. First, run the following helper code cell (no output):

In [ ]:
# HELPERS

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import RegularPolygon, Circle
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import matplotlib.image as mpimg  # for background images

def draw_hexagonal_cells(
    cell_radius_m,
    grid_extent_m=3000,
    city_diameter_m=5000,
    city_center=(0.0, 0.0),   # in the same meter coords as the grid
    user_density=5,           # just to set the title
    highlight_radius_m=10000,  # optional: draw these cells in green, others in blue
    face_alpha=.7,
    # background options
    background_fit="axes",        # "axes" or "city"
    background_alpha=1.0,         # image transparency
    background_origin="upper",    # "upper" or "lower"
    background_image_path="../../.admin/notebooks/picture/zurich_map.jpg",  # your map
):
    cx, cy = city_center
    city_R = city_diameter_m / 2.0

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_aspect('equal')

    # ---- Background image (under everything) ----
    if background_image_path is not None:
        try:
            img = mpimg.imread(background_image_path)
            if background_fit == "city":
                extent = [cx - city_R, cx + city_R, cy - city_R, cy + city_R]
            else:  # "axes"
                extent = [-grid_extent_m, grid_extent_m, -grid_extent_m, grid_extent_m]
            ax.imshow(img, extent=extent, origin=background_origin, alpha=background_alpha, zorder=0)
        except Exception as e:
            print(f"[warn] Could not load background image: {e}")

    dx = 1.5 * cell_radius_m
    dy = np.sqrt(3) * cell_radius_m

    nx = int(np.ceil(grid_extent_m / dx)) + 2
    ny = int(np.ceil(grid_extent_m / dy)) + 2

    # Counters for legend
    normal_count = 0
    highlight_count = 0

    for i in range(-nx, nx + 1):
        col_offset = (i % 2) * (dy / 2.0)
        for j in range(-ny, ny + 1):
            x = i * dx
            y = j * dy + col_offset

            # keep only hex centers inside the circular boundary
            if (x - cx)**2 + (y - cy)**2 <= city_R**2:
                if highlight_radius_m is not None and np.isclose(cell_radius_m, highlight_radius_m):
                    hex_color = 'green'
                    highlight_count += 1
                else:
                    hex_color = 'blue'
                    normal_count += 1

                hexagon = RegularPolygon(
                    (x, y),
                    numVertices=6,
                    radius=cell_radius_m,
                    orientation=np.radians(30),
                    facecolor=hex_color,
                    alpha=face_alpha,
                    edgecolor='black',
                    linewidth=0.5,
                    label=None  # avoid per-cell legend entries
                )
                hexagon.set_zorder(2)
                ax.add_patch(hexagon)

    # City boundary
    boundary = Circle(
        (cx, cy),
        radius=city_R,
        fill=False,
        edgecolor='red',
        linewidth=2.0
    )
    boundary.set_zorder(3)
    ax.add_patch(boundary)

    # Axes, labels, title
    ax.set_xlim(-grid_extent_m, grid_extent_m)
    ax.set_ylim(-grid_extent_m, grid_extent_m)
    ax.set_facecolor((0.2, 0.2, 0.3))
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.grid(True, linestyle='--', alpha=.3)

    # --- Custom legend with counts ---
    handles = []
    if normal_count > 0:
        handles.append(mpatches.Patch(facecolor='blue', edgecolor='black',
                                      label=f"{normal_count} cell(s)"))
    if highlight_count > 0:
        handles.append(mpatches.Patch(facecolor='green', edgecolor='black',
                                      label=f"{highlight_count} cell(s)"))
    # Boundary proxy (red line) for legend
    handles.append(Line2D([0], [0], color='red', lw=2, label="City of Zurich"))

    ax.legend(handles=handles, fancybox=True, framealpha=1, loc="upper right",
              facecolor=(0.6, 0.61, 0.8))

    plt.tight_layout()
    plt.show()

# Functions to compute coverage and capacity-limited radius
def coverage_radius(tx_power, rx_sensitivity, frequency, antenna_gain):
    fspl = tx_power + 2 * antenna_gain - rx_sensitivity  # Free-space path loss
    wavelength = speed_of_light / frequency
    radius = 10 ** ((fspl - 20 * np.log10(4 * np.pi / wavelength)) / 20)
    return radius

def capacity_radius(user_density, data_rate_per_user, bandwidth, sinr_linear):
    total_demand = user_density * data_rate_per_user  # bps per km^2
    cell_capacity = bandwidth * np.log2(1 + sinr_linear)  # bps
    if total_demand>0:
        cell_area = cell_capacity / total_demand  # km^2
    else:
        cell_area = 1e12; # extremely high, therefore not relevant
    radius = np.sqrt(cell_area / np.pi) * 1000  # Convert to meters
    return radius


def network_planning_summary():
    """
    Prints a formatted summary using the global variables:
      - tx_power               (dBm)
      - user_density           (users/km²)
      - data_rate_per_user     (bit/s)
      - bandwidth              (Hz)

    Call with no arguments after defining those names somewhere in your notebook.
    """
    need = ["tx_power", "user_density", "data_rate_per_user", "bandwidth"]
    g = globals()
    missing = [n for n in need if n not in g]
    if missing:
        raise NameError(
            "Missing required variables: " + ", ".join(missing) +
            ". Please define them (e.g., tx_power=23.0, user_density=1500, data_rate_per_user=5e6, bandwidth=20e6)."
        )

    tx_power = g["tx_power"]
    user_density = g["user_density"]
    data_rate_per_user = g["data_rate_per_user"]
    bandwidth = g["bandwidth"]
    
    print(f"(1) Transmit power (Tx): {tx_power:.2f} dBm")
    print(f"(2) User density: {user_density:.2f} users/km²")
    print(f"(3) Target data rate per user: {data_rate_per_user/1e6:.2f} Mb/s")
    print(f"(4) Channel bandwidth: {bandwidth/1e6:.2f} MHz")
    

def compute_radio_planning(update_globals: bool = True):
    """
    Uses globals already defined in the notebook (no checks) to compute
    link budget & capacity/coverage metrics, and returns a dict.

    Expects these globals to exist:
      tx_power, user_density, data_rate_per_user, bandwidth,
      frequency, speed_of_light, rx_sensitivity, noise_power_density,
      antenna_gain, sinr_target,
      db_to_linear, linear_to_db, fspl_db,
      coverage_radius, capacity_radius
    """
    g = globals()

    # pull from globals (assumes they exist)
    tx_power            = g["tx_power"]
    user_density        = g["user_density"]
    data_rate_per_user  = g["data_rate_per_user"]
    bandwidth           = g["bandwidth"]
    frequency           = g["frequency"]
    speed_of_light      = g["speed_of_light"]
    rx_sensitivity      = g["rx_sensitivity"]
    noise_power_density = g["noise_power_density"]
    antenna_gain        = g["antenna_gain"]
    sinr_target         = g["sinr_target"]

    db_to_linear_fn     = g["db_to_linear"]
    linear_to_db_fn     = g["linear_to_db"]
    fspl_db_fn          = g["fspl_db"]

    coverage_radius_fn  = g["coverage_radius"]
    capacity_radius_fn  = g["capacity_radius"]

    import math, numpy as np

    # wavelength
    wavelength = speed_of_light / frequency

    # FSPL helpers
    f_MHz = frequency / 1e6
    fspl_100m = fspl_db_fn(f_MHz, 0.1)  # 100 m
    fspl_1km  = fspl_db_fn(f_MHz, 1.0)  # 1 km

    # Thermal noise floor over the occupied bandwidth (ideal)
    noise_floor_dbm = noise_power_density + 10 * math.log10(bandwidth)

    # Required received power to hit the SINR target (assuming negligible interference)
    required_rx_power_dbm = noise_floor_dbm + sinr_target

    # Simple link budget (same antenna gain at Tx and Rx; no other losses)
    total_antenna_gain = antenna_gain + antenna_gain  # dB
    rx_power_100m = tx_power + total_antenna_gain - fspl_100m
    rx_power_1km  = tx_power + total_antenna_gain - fspl_1km

    margin_100m = rx_power_100m - required_rx_power_dbm
    margin_1km  = rx_power_1km  - required_rx_power_dbm

    # Max free-space range where link budget meets the SINR target (very idealized)
    fspl_max = tx_power + total_antenna_gain - required_rx_power_dbm
    d_km_max = 10 ** ((fspl_max - 32.44 - 20*math.log10(f_MHz)) / 20)

    # Shannon capacity at SINR target (upper bound)
    sinr_linear = db_to_linear_fn(sinr_target)
    shannon_spectral_eff = math.log2(1 + sinr_linear)       # bits/s/Hz
    shannon_capacity_bps = bandwidth * shannon_spectral_eff # bps

    # Rough demand & users supported at target rate
    per_km2_demand_bps = user_density * data_rate_per_user
    users_supported_at_target = shannon_capacity_bps / data_rate_per_user

    # Alternative noise power (mW) if you need it in linear
    noise_power_mw = 10 ** ((noise_power_density + 10 * np.log10(bandwidth)) / 10)

    # Coverage & capacity radii (your own functions)
    d_coverage = coverage_radius_fn(tx_power, rx_sensitivity, frequency, antenna_gain)
    d_capacity = capacity_radius_fn(user_density, data_rate_per_user, bandwidth, sinr_linear)
    d_cell = min(d_coverage, d_capacity)

    results = {
        "wavelength_m": wavelength,
        "f_MHz": f_MHz,
        "fspl_100m_db": fspl_100m,
        "fspl_1km_db": fspl_1km,
        "noise_floor_dbm": noise_floor_dbm,
        "required_rx_power_dbm": required_rx_power_dbm,
        "total_antenna_gain_db": total_antenna_gain,
        "rx_power_100m_dbm": rx_power_100m,
        "rx_power_1km_dbm": rx_power_1km,
        "margin_100m_db": margin_100m,
        "margin_1km_db": margin_1km,
        "fspl_max_db": fspl_max,
        "d_km_max": d_km_max,
        "sinr_linear": sinr_linear,
        "shannon_spectral_eff_bphz": shannon_spectral_eff,
        "shannon_capacity_bps": shannon_capacity_bps,
        "per_km2_demand_bps": per_km2_demand_bps,
        "users_supported_at_target": users_supported_at_target,
        "noise_power_mw": noise_power_mw,
        "d_coverage_m": d_coverage,
        "d_capacity_m": d_capacity,
        "d_cell_m": d_cell,
    }

    if update_globals:
        g.update({
            "wavelength": wavelength,
            "f_MHz": f_MHz,
            "fspl_100m": fspl_100m,
            "fspl_1km": fspl_1km,
            "noise_floor_dbm": noise_floor_dbm,
            "required_rx_power_dbm": required_rx_power_dbm,
            "total_antenna_gain": total_antenna_gain,
            "rx_power_100m": rx_power_100m,
            "rx_power_1km": rx_power_1km,
            "margin_100m": margin_100m,
            "margin_1km": margin_1km,
            "fspl_max": fspl_max,
            "d_km_max": d_km_max,
            "shannon_spectral_eff": shannon_spectral_eff,
            "shannon_capacity_bps": shannon_capacity_bps,
            "per_km2_demand_bps": per_km2_demand_bps,
            "users_supported_at_target": users_supported_at_target,
            "noise_power": noise_power_mw,
            "sinr_linear": sinr_linear,
            "d_coverage": d_coverage,
            "d_capacity": d_capacity,
            "d_cell": d_cell,
        })
    return results



frequency = 2.4e9          # Hz (e.g., 2.4 GHz)
speed_of_light = 3e8       # m/s
rx_sensitivity = -80       # dBm
noise_power_density = -174 # dBm/Hz (thermal noise at ~290K)
antenna_gain = 0           # dBi (per end; simple assumption)
sinr_target = 10           # dB (target SINR)

import math

def db_to_linear(db): 
    return 10**(db/10)

def linear_to_db(x): 
    return 10*math.log10(x)

# Free-space path loss (FSPL) helper: frequency in MHz, distance in km
def fspl_db(f_MHz, d_km):
    return 32.44 + 20*math.log10(f_MHz) + 20*math.log10(d_km)



#### 2.3.1 Scenario parameters

Let's define some important scenario parameters, starting with the maximum transmission power.

##### 2.3.1.1 `tx_power`

A common number for the transmission power is $23\,\text{dBm} ≈ 0.200\,\text{W}$ (about $200\,\text{mW}$).

$$
P_{\text{TX}} = 10^{(\text{dBm}-30)/10} = 10^{(23-30)/10} = 10^{-0.7} \approx 0.1995\ \text{W}
$$

In [ ]:
tx_power = 23              # dBm

##### 2.3.1.2 `user_density`

We use Zurich as a playground.  With the city of Zurich covering roughly 75 km² and about 500'000 people, the average user density is ≈ 7000 users/km²; in an extreme case like the Zurich Street Parade, with an additional 800'000 people (≈ 1.3 million total), the density rises to ≈ 17000 users/km². It is defined as the average number of users per km².

In [ ]:
# number_of_Zuerchers = 1     # extreme case, only one user in the whole city
number_of_Zuerchers = 0.5e6     # normal weekday
#number_of_Zuerchers = 1.3e6    # street parade - “Hopp Züri!”

user_density = number_of_Zuerchers / 75

##### 2.3.1.3 `data_rate_per_user`

This determines the requirement per user, in bits per second ($\text{b/s}$). A common number is $1\,\text{Mb/s}$.

In [ ]:
data_rate_per_user = .5e6   # for example, 1e6 = 1000000 = 1 Mb/s per user

##### 2.3.1.4 `bandwidth`

The system bandwidth in $\text{Hz}$ determines the available spectrum for each cell.

In [ ]:
bandwidth = 100e6           # Hz 

#### 2.3.2 Visualization result

Run the computation below and observe the planning result.

In [ ]:
compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)

----
### 2.4 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**What is the benefit of a small cell with a smaller coverage area? What is the disadvantage?** Answer:

(add your answer here)

**Why are there so many cell towers in Zurich? Why can a telecom operator not work with just one antenna, say Üetliberg, and serve all cellphones in the area?** Answer:

(add your answer here)

**What effect does increasing the carrier frequency have on a cell, in (a) an outdoor free space scenario, and (b) in an indoor scenario with obstacles?** Answer:

(add your answer here)

**One Zuercher in the city: Run the following scenario and explain why the network still requires more than one cell, despite the fact that there is only one user. How does increasing the bandwidth or the transmission power make a difference? Explain why!**

In [ ]:
tx_power = 23                                # dBm
number_of_Zuerchers = 1                   # extreme case, only one user in the whole city
user_density = number_of_Zuerchers / 75
data_rate_per_user = .5e6
bandwidth = 100e6 

compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)

Answer: (add your answer here)

**Now run the following Street Parade scenario. You see a large number of required cells. Explain this higher number. How does increasing the bandwidth or the transmission power make a difference?**

In [ ]:
tx_power = 23                                # dBm
number_of_Zuerchers = 1.3e6                  # street parade - “Hopp Züri!”
user_density = number_of_Zuerchers / 75
data_rate_per_user = .5e6
bandwidth = 100e6 

compute_radio_planning()
network_planning_summary()
draw_hexagonal_cells(d_cell)


Answer: (add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)